# 🧠🔥 NV 3 OMEGA — Google Colab Pro A100 超大規模・感情共感型人間NPC 学習スタジオ

**ワンクリック実行ガイド**: 上部メニューの **`ランタイム` → `すべてのセルを実行` (Ctrl+F9 / ⌘+F9)** を押すだけで全自動完了します！
（※万が一セル2単体から実行しても自動でリポジトリを取得・復旧する自己修復構成です）

### NV 3 が NV2.6 Force / RNV1 FAST を圧倒する 6 大革新的アーキテクチャ
1. **419万パラメータ A100 Omega Teacher (`NV3OmegaBrain`)**: 2層 SwiGLU Entity-Relation Transformer (12エンティティ関係推論) + 2層 cuDNN融合 GRU-384 + 4層 512次元 Centralized Critic (CTDE) で A100 Tensor Core (BF16) をフル稼働！
2. **ゼロ同期 Pure-CUDA 超並列シミュレーション (`HarborOmegaEnvGPU`)**: A100-40GBで **1,024アリーナ = 6,144体**、A100-80GBで **2,048アリーナ = 12,288体** の感情を持つ人間NPCが同時に戦闘・共感・学習！
3. **8次元リアル感情 + 4次元身体自己認識 + 16次元パーソナリティDNA**: 冷静・恐怖・勇気・極限集中・焦り・信頼・好奇心・安堵、心拍数(BPM)・疲労・被弾リスク・自信が射撃精度・移動・遮蔽・深呼吸リセット・ハンドサイン・Klatt音声無線に直結！
4. **Factored 6-Group PPO Clipping + 5大自己教師あり理解損失**: RNV1の多頭クリッピング枯渇を完全解消し、感情評価・危険察知・6セクター敵予測(Friston)・敵の意図予測(Theory of Mind)・味方共感クロスファイアを同時学習！
5. **4フェーズ極限カリキュラム + 敵対的 Counter-Exploiter**: PFSPリーグ(20世代)に加え、NV 3の弱点だけを突く専用敵対AI(Exploiter)を同時育成して無敵の戦術柔軟性を獲得！
6. **Phase 4 Teacher → 360Hz Browser Core 蒸留**: 419万パラメータの頭脳を 58.8万パラメータ (`~800 KB` Int8 `nv3-weights.json`) へ高精度DAgger蒸留し、そのままブラウザで 360Hz 動作！

> **推奨ランタイム**: `ランタイム` → `ランタイムのタイプを変更` → **A100 GPU (ハイメモリ)**

In [ ]:
#@title 🚀 セル1: 環境セットアップ & A100 ハードウェア/パリティ自動検証
REPO_URL = "https://github.com/sc8z35a-collab/Result.git" #@param {type:"string"}
REPO_BRANCH = "auto" #@param ["auto", "feat/rnv1-colab-a100-studio", "main"]
SAVE_TO_DRIVE = False #@param {type:"boolean"}
DRIVE_SUBDIR = "Result_NV3_Omega_Studio" #@param {type:"string"}

import importlib, os, shutil, subprocess, sys
from pathlib import Path
import torch

REPO_DIR = Path("/content/Result")
OUT_DIR = Path("/content/nv3_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, cwd=None):
    res = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"❌ Command failed ({res.returncode}): {" ".join(map(str, cmd))}")
        if res.stdout: print("[stdout]", res.stdout[-1500:])
        if res.stderr: print("[stderr]", res.stderr[-1500:])
        raise subprocess.CalledProcessError(res.returncode, cmd, res.stdout, res.stderr)
    return res

def ensure_nv3_repo(repo_url=REPO_URL, pref_branch=REPO_BRANCH):
    if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
        shutil.rmtree(REPO_DIR)
    if not REPO_DIR.exists():
        run_cmd(["git", "clone", "--depth", "1", repo_url, str(REPO_DIR)])
    else:
        run_cmd(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", repo_url])

    def try_branch(br):
        r = subprocess.run(["git", "-C", str(REPO_DIR), "ls-remote", "--heads", "origin", f"refs/heads/{br}"], capture_output=True, text=True)
        if r.returncode != 0 or not r.stdout.strip():
            return False
        run_cmd(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", f"+refs/heads/{br}:refs/remotes/origin/{br}"])
        run_cmd(["git", "-C", str(REPO_DIR), "checkout", "-B", br, "FETCH_HEAD"])
        run_cmd(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"])
        return (REPO_DIR / "training" / "nv3_omega_studio" / "trainer_nv3.py").exists()

    candidates = ["feat/rnv1-colab-a100-studio", "main"] if pref_branch == "auto" else [pref_branch, "feat/rnv1-colab-a100-studio", "main"]
    selected = None
    for br in candidates:
        if try_branch(br):
            selected = br
            break
    assert selected is not None, "❌ training/nv3_omega_studio/trainer_nv3.py がリポジトリに見つかりません。"
    os.chdir(REPO_DIR)
    repo_str = str(REPO_DIR)
    if repo_str in sys.path:
        sys.path.remove(repo_str)
    sys.path.insert(0, repo_str)
    importlib.invalidate_caches()
    return selected

SELECTED_BRANCH = ensure_nv3_repo()
print(f"✅ リポジトリ準備完了: {REPO_DIR} (branch={SELECTED_BRANCH})")

# Optional Google Drive auto-backup & seamless resume
DRIVE_DIR = None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_DIR = str(Path("/content/drive/MyDrive") / DRIVE_SUBDIR)
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
    print(f"✅ Google Drive 自動バックアップ先: {DRIVE_DIR}")

# Verify GPU (A100 Tensor Cores / BF16)
assert torch.cuda.is_available(), "❌ GPU が有効になっていません！ランタイム → ランタイムのタイプを変更 → A100 GPU を選択してください。"
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024 ** 3)
print(f"✅ GPU 検出: {props.name} | VRAM: {vram_gb:.1f} GB | BF16: {torch.cuda.is_bf16_supported()} | PyTorch: {torch.__version__}")
if "A100" not in props.name and "H100" not in props.name:
    print("⚠️ A100 以外の GPU です。自動でバッチサイズを最適調整して続行します。")

# Verify import right inside the Colab Python kernel
from training.nv3_omega_studio.trainer_nv3 import NV3OmegaConfig, NV3_PRESETS, train_nv3_omega
print("✅ training.nv3_omega_studio インポート確認OK")

# Ensure Node.js >= 18 for PyTorch <-> Browser JS parity & real 360Hz arena gauntlet
def get_node_major():
    try:
        r = subprocess.run(["node", "-v"], capture_output=True, text=True)
        if r.returncode == 0:
            v = r.stdout.strip().lstrip("v")
            return int(v.split(".")[0]), r.stdout.strip()
    except Exception:
        pass
    return 0, "none"

node_major, node_str = get_node_major()
if node_major < 18:
    print(f"📦 Node.js ({node_str}) -> Node.js 20 LTS をインストール中...")
    subprocess.run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y -qq nodejs", shell=True, check=False)
    node_major, node_str = get_node_major()
print(f"✅ Node.js: {node_str}")

if node_major >= 18:
    p_res = run_cmd([sys.executable, "-m", "training.nv3_omega_studio.test_nv3_parity"], cwd=REPO_DIR)
    for ln in p_res.stdout.strip().splitlines():
        print("  ", ln)
else:
    print("⚠️ Node.js 18+ 未検出のため JS パリティ検証をスキップして学習へ進みます。")

print("🎉 セットアップ完了！そのままセル2で NV 3 OMEGA 超大規模学習を開始します。")


In [ ]:
#@title 🧠 セル2: NV 3 OMEGA 4フェーズ超大規模A100学習 + 感情/戦術ダッシュボード + ブラウザ蒸留
# プリセット目安 (A100 Pro):
#  - a100_heavy_1h   : 3億ステップ (~40-60分) — 圧倒的強さ＆自然な感情・人間性 (おすすめ！)
#  - a100_extreme_3h : 10億ステップ (~2.5-3.5時間) — プロ級チーム連携・ベイト・ToM極致
#  - a100_omega_6h   : 20億ステップ (~5-6時間) — 究極の超リアル人間NPC (Drive保存推奨)
#  - a100_verify_5m  : 2,500万ステップ (~3-5分) — 動作確認＆超高速テスト用
PRESET = "a100_heavy_1h" #@param ["a100_heavy_1h", "a100_extreme_3h", "a100_omega_6h", "a100_verify_5m"]
QUANTIZE = "q8" #@param ["q8", "f32"]
FRESH_START = False #@param {type:"boolean"}
SEED = 2026 #@param {type:"integer"}

import copy, importlib, os, shutil, subprocess, sys
from pathlib import Path
from IPython.display import Image, display

# Self-healing bootstrap: works even if Cell 2 is run standalone or after a runtime restart!
REPO_DIR = Path("/content/Result")
if not (REPO_DIR / "training" / "nv3_omega_studio" / "trainer_nv3.py").exists():
    print("⚙️ リポジトリ未取得を検出 -> 自動セットアップ中...")
    if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
        shutil.rmtree(REPO_DIR)
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sc8z35a-collab/Result.git", str(REPO_DIR)], check=True)
    for br in ("feat/rnv1-colab-a100-studio", "main"):
        r = subprocess.run(["git", "-C", str(REPO_DIR), "ls-remote", "--heads", "origin", f"refs/heads/{br}"], capture_output=True, text=True)
        if r.returncode == 0 and r.stdout.strip():
            subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", f"+refs/heads/{br}:refs/remotes/origin/{br}"], check=True)
            subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", br, "FETCH_HEAD"], check=True)
            subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)
            if (REPO_DIR / "training" / "nv3_omega_studio" / "trainer_nv3.py").exists():
                break

os.chdir(REPO_DIR)
if str(REPO_DIR) in sys.path:
    sys.path.remove(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))
importlib.invalidate_caches()

from training.nv3_omega_studio.trainer_nv3 import NV3OmegaConfig, NV3_PRESETS, train_nv3_omega

OUT_DIR = Path("/content/nv3_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
drive_dir = globals().get("DRIVE_DIR", None)

p = copy.deepcopy(NV3_PRESETS[PRESET])
cfg = NV3OmegaConfig(
    preset=PRESET,
    output_dir=str(OUT_DIR),
    drive_dir=drive_dir,
    quantize=QUANTIZE,
    fresh_start=FRESH_START,
    seed=SEED,
    **p,
)

result = train_nv3_omega(cfg)
WEIGHTS_PATH = Path(result["weights_path"])
DASH_PATH = Path(result["dashboard_path"])

print(f"\n✅ NV 3 蒸留＆エクスポート完了: {WEIGHTS_PATH} ({WEIGHTS_PATH.stat().st_size / 1024:.1f} KB)")
if DASH_PATH.exists():
    display(Image(filename=str(DASH_PATH)))


In [ ]:
#@title ⚔️ セル3: 実ゲーム 360Hz エンジン総当たり戦 (NV 3 vs NV2.6 Force / RNV1 FAST / NV2.5 / NV2.1 / NV1)
MATCHES_PER_OPPONENT = 4 #@param {type:"integer"}
SECONDS_PER_MATCH = 40 #@param {type:"integer"}
WEAPON = "MS10" #@param ["MS10", "SW T", "SRA", "D1", "RD20", "RD30", "RD40", "DF1 OUT"]

import shutil, subprocess
from pathlib import Path

REPO_DIR = Path("/content/Result")
WEIGHTS_PATH = Path(globals().get("WEIGHTS_PATH", Path("/content/nv3_outputs/nv3-weights.json")))

if not WEIGHTS_PATH.exists():
    print(f"⚠️ 学習済み重みファイルがまだ存在しません: {WEIGHTS_PATH}")
    print("👉 先にセル2（NV 3 OMEGA 学習）を完了させてからセル3を実行してください。")
elif not shutil.which("node"):
    print("⚠️ Node.js が見つからないため実機JSベンチマークをスキップします（セル1を実行すると自動導入されます）。")
else:
    cmd = [
        "node",
        "training/nv3_omega_studio/eval_nv3_arena.mjs",
        "--weights", str(WEIGHTS_PATH),
        "--opponents", "human,rnv1,humain,nv21,classic,nv1",
        "--matches", str(MATCHES_PER_OPPONENT),
        "--seconds", str(SECONDS_PER_MATCH),
        "--weapon", WEAPON,
    ]
    subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


In [ ]:
#@title ⬇️ セル4: `nv3-weights.json` を自動ダウンロード（ゲームに1クリック読込！）
from pathlib import Path

WEIGHTS_PATH = Path(globals().get("WEIGHTS_PATH", Path("/content/nv3_outputs/nv3-weights.json")))
drive_dir = globals().get("DRIVE_DIR", None)

assert WEIGHTS_PATH.exists(), f"❌ 重みファイルが見つかりません: {WEIGHTS_PATH}（先にセル2の学習を完了させてください）"
print(f"📦 ダウンロード準備完了: {WEIGHTS_PATH.name} ({WEIGHTS_PATH.stat().st_size / 1024:.1f} KB)")
if drive_dir:
    print(f"☁️ Google Drive にも保存済: {Path(drive_dir) / WEIGHTS_PATH.name}")

print("""
=== ゲーム (Result) への導入方法 ===
1. Result の対戦設定ロビーで「実験場」を開く
2. 紫色の NV 3 バーにある「NV3 重み読込 (.json)」を押して `nv3-weights.json` を選択
3. 自動で通常戦 & 実験場に『NV 3』が適用され、感情・呼吸・ハンドサイン・無線ボイス付きで 360Hz 稼働します！
""")

try:
    from google.colab import files
    files.download(str(WEIGHTS_PATH))
except Exception as e:
    print(f"(Colab 外実行のため自動ダウンロードをスキップしました: {e})")
